# Démonstration Intégrée : Biologie + Transport Distribué

Ce notebook démontre le système complet avec :

-   **Biologie** : Croissance exponentielle distribuée sur CellWorkers
-   **Transport** : Advection + diffusion centralisée sur TransportWorker
-   **Workflow** : EventScheduler coordonnant les 5 phases

**Configuration** :

-   Grille 100×100 (1000km × 1000km)
-   4 workers (2×2)
-   Courant vers l'EST : u=0.2 m/s
-   Diffusion : D=500 m²/s
-   Île centrale bloquant le flux
-   Conditions limites CLOSED


In [ ]:
# Imports
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import ray

from seapopym_message.core.kernel import Kernel
from seapopym_message.core.unit import unit
from seapopym_message.distributed.scheduler import EventScheduler
from seapopym_message.distributed.worker import CellWorker2D
from seapopym_message.transport.worker import TransportWorker
from seapopym_message.utils.grid import GridInfo

print("✅ Modules importés avec succès !")

## 1. Configuration du système distribué


In [ ]:
# Initialiser Ray
if not ray.is_initialized():
    ray.init(num_cpus=2, num_gpus=2)
    print("✅ Ray initialisé")
else:
    print("✅ Ray déjà initialisé")

In [ ]:
# Paramètres de la grille globale
global_nlat, global_nlon = 100, 100
dx = 10e3  # 10 km
dy = 10e3  # 10 km

# Configuration des workers
n_workers_lat, n_workers_lon = 2, 2
nlat_per_worker = global_nlat // n_workers_lat
nlon_per_worker = global_nlon // n_workers_lon

print(f"Grille globale: {global_nlat}×{global_nlon}")
print(f"Domaine: {global_nlat * dy / 1e3:.0f} km × {global_nlon * dx / 1e3:.0f} km")
print(f"Workers: {n_workers_lat}×{n_workers_lon} = {n_workers_lat * n_workers_lon} total")
print(f"Taille par worker: {nlat_per_worker}×{nlon_per_worker}")

## 2. Définition de l'unité de biologie


In [ ]:
GROWTH_RATE = 0.1 / 14400


# Unité de croissance exponentielle simple
@unit(
    name="exponential_growth",
    inputs=["biomass"],
    outputs=["biomass"],
    scope="local",
    compiled=False,
)
def exponential_growth(biomass, dt, params, forcings=None):
    """Croissance exponentielle: dB/dt = r*B."""
    r = params.get("growth_rate", GROWTH_RATE)
    biomass_new = biomass * jnp.exp(-r * dt)
    return biomass_new


# Créer le kernel
kernel = Kernel([exponential_growth])
params = {"growth_rate": GROWTH_RATE}  # Très faible croissance pour voir surtout le transport

print("✅ Kernel biologique créé")

## 3. Création de la biomasse initiale et de l'île


In [ ]:
# Créer le blob de biomasse initial (gaussien)
x = np.arange(global_nlon)
y = np.arange(global_nlat)
X, Y = np.meshgrid(x, y)

# Position initiale du blob (à gauche)
blob_x0, blob_y0 = 20, 50
blob_sigma = 10

biomass_init_global = 100.0 * np.exp(
    -((X - blob_x0) ** 2 + (Y - blob_y0) ** 2) / (2 * blob_sigma**2)
)

# Créer l'île (masque)
island_x, island_y = 50, 50
island_radius = 15

mask_global = np.ones((global_nlat, global_nlon), dtype=np.float32)
distance_to_island = np.sqrt((X - island_x) ** 2 + (Y - island_y) ** 2)
mask_global[distance_to_island <= island_radius] = 0.0

# Convertir en JAX
biomass_init_global = jnp.array(biomass_init_global, dtype=jnp.float32)
mask_global_jax = jnp.array(mask_global, dtype=jnp.float32)

# Visualiser
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im1 = axes[0].imshow(biomass_init_global, origin="lower", cmap="YlOrRd", interpolation="bilinear")
axes[0].set_title("Biomasse Initiale [kg/m²]")
axes[0].set_xlabel("Longitude (index)")
axes[0].set_ylabel("Latitude (index)")
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(mask_global, origin="lower", cmap="gray_r", interpolation="nearest")
axes[1].set_title("Masque (blanc=océan, noir=île)")
axes[1].set_xlabel("Longitude (index)")
axes[1].set_ylabel("Latitude (index)")
plt.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow(
    biomass_init_global * mask_global, origin="lower", cmap="YlOrRd", interpolation="bilinear"
)
axes[2].contour(mask_global, levels=[0.5], colors="black", linewidths=2)
axes[2].set_title("Biomasse + Île")
axes[2].set_xlabel("Longitude (index)")
axes[2].set_ylabel("Latitude (index)")
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

mass_init = float(jnp.sum(biomass_init_global * mask_global_jax * dx * dy))
print(f"Masse initiale : {mass_init:.2e} kg")

## 4. Création des CellWorkers


In [ ]:
# GridInfo pour les workers
grid_info = GridInfo(
    lat_min=-500,  # km (arbitraire pour grille plane)
    lat_max=500,
    lon_min=-500,
    lon_max=500,
    nlat=global_nlat,
    nlon=global_nlon,
)

# Créer les workers
workers = []
for i in range(n_workers_lat):
    for j in range(n_workers_lon):
        lat_start = i * nlat_per_worker
        lat_end = (i + 1) * nlat_per_worker
        lon_start = j * nlon_per_worker
        lon_end = (j + 1) * nlon_per_worker

        worker = CellWorker2D.remote(
            worker_id=i * n_workers_lon + j,
            grid_info=grid_info,
            lat_start=lat_start,
            lat_end=lat_end,
            lon_start=lon_start,
            lon_end=lon_end,
            kernel=kernel,
            params=params,
        )

        # Initialiser l'état avec la portion correspondante de biomasse globale
        biomass_patch = biomass_init_global[lat_start:lat_end, lon_start:lon_end]
        ray.get(worker.set_initial_state.remote({"biomass": biomass_patch}))

        workers.append(worker)

print(f"✅ {len(workers)} CellWorkers créés et initialisés")

## 5. Création du TransportWorker


In [ ]:
# Créer le TransportWorker (grille plane)
transport_worker = TransportWorker.remote(
    grid_type="plane",
    nlat=global_nlat,
    nlon=global_nlon,
    dx=dx,
    dy=dy,
    lat_bc="closed",
    lon_bc="closed",
)

print("✅ TransportWorker créé")

## 6. Préparation des forcings de transport


In [ ]:
# Champs de vitesse (courant vers l'EST)
u_global = jnp.ones((global_nlat, global_nlon), dtype=jnp.float32) * 0.2  # 0.2 m/s vers l'est
v_global = jnp.zeros((global_nlat, global_nlon), dtype=jnp.float32)  # Pas de déplacement N-S

# Coefficient de diffusion
D = 500.0  # m²/s

# Pas de temps
dt = 1 * 60 * 60

# Vérifier CFL
cfl_adv = 0.2 * dt / dx
cfl_diff = D * dt / dx**2

print(f"Vitesse : u=0.2 m/s (EST), v=0.0 m/s")
print(f"Diffusion : D={D:.1f} m²/s")
print(f"Pas de temps : dt={dt:.0f} s = {dt / 60:.1f} min")
print(f"\nCFL advection : {cfl_adv:.4f} (< 1 ✅)")
print(f"CFL diffusion : {cfl_diff:.4f} (< 0.25 ✅)")

## 7. Création de l'EventScheduler et simulation


In [ ]:
# Créer le scheduler avec transport activé
# On va simuler ~16.7 heures comme dans l'exemple transport_demo_island.ipynb
t_total = 30 * 4 * 60 * 60

# Créer un ForcingManager simple pour injecter u, v, D, mask
# On va utiliser ray.put pour mettre les forcings en mémoire partagée
forcings_global_ref = ray.put(
    {
        "u": u_global,
        "v": v_global,
        "D": D,
        "ocean_mask": mask_global_jax,
    }
)

print(f"✅ Forcings mis en mémoire partagée Ray")
print(f"Durée simulation : {t_total / 3600:.2f} heures")

In [ ]:
# Wrapper pour ForcingManager simulé
class SimpleForcingManager:
    """Manager simple qui retourne toujours les mêmes forcings."""

    def __init__(self, forcings_ref):
        self.forcings_ref = forcings_ref

    def prepare_timestep_distributed(self, time, params):
        """Retourne l'ObjectRef des forcings."""
        return self.forcings_ref


forcing_manager = SimpleForcingManager(forcings_global_ref)

# Créer le scheduler
scheduler = EventScheduler(
    workers=workers,
    dt=dt,
    t_max=t_total,
    forcing_manager=forcing_manager,
    transport_worker=transport_worker,
    transport_enabled=True,
    global_nlat=global_nlat,
    global_nlon=global_nlon,
    forcing_params={"horizontal_diffusivity": D},
)

print("✅ EventScheduler créé avec transport activé")

In [ ]:
# Nombre de snapshots à sauvegarder (modifiable)
N_intermediate = 4  # Nombre d'états intermédiaires
N_snapshots_total = N_intermediate + 2  # + début + fin

# Calculer les timesteps où sauvegarder
n_steps_total = int(t_total / dt)
snapshot_steps = np.linspace(0, n_steps_total, N_snapshots_total, dtype=int)

print(f"Nombre total de timesteps : {n_steps_total}")
print(f"Snapshots sauvegardés à : {snapshot_steps}")
print(f"Temps correspondants : {snapshot_steps * dt / 3600} heures")

# Stockage des snapshots
biomass_snapshots = []
time_snapshots = []
conservation_history = []
mass_history = []

# Sauvegarder l'état initial (t=0)
biomass_snapshots.append(np.array(biomass_init_global))
time_snapshots.append(0.0)
mass_history.append(mass_init)
conservation_history.append(100.0)

print("\n🚀 Début de la simulation...")

# Exécuter la simulation step par step
step = 0
next_snapshot_idx = 1

while scheduler.t_current < t_total - 1e-6:
    # Exécuter un step
    diagnostics = scheduler.step()
    step += 1

    # Vérifier si on doit sauvegarder un snapshot
    if next_snapshot_idx < len(snapshot_steps) and step >= snapshot_steps[next_snapshot_idx]:
        # Collecter la biomasse globale
        biomass_global_current = scheduler._collect_global_biomass()
        biomass_snapshots.append(np.array(biomass_global_current))
        time_snapshots.append(scheduler.t_current / 3600)  # heures

        # Conservation
        if "transport" in diagnostics:
            mass_current = diagnostics["transport"]["mass_after"]
            conservation = diagnostics["transport"]["conservation_fraction"] * 100
        else:
            # Calculer manuellement si pas de transport
            mass_current = float(jnp.sum(biomass_global_current * mask_global_jax * dx * dy))
            conservation = mass_current / mass_init * 100

        mass_history.append(mass_current)
        conservation_history.append(conservation)

        print(
            f"  Snapshot {next_snapshot_idx}/{N_snapshots_total - 1} - "
            f"Step {step}/{n_steps_total} - t={time_snapshots[-1]:.2f}h - "
            f"Conservation: {conservation:.4f}%"
        )

        next_snapshot_idx += 1

print("\n✅ Simulation terminée !")
print(f"Conservation finale : {conservation_history[-1]:.6f}%")
print(f"Perte/Gain de masse : {conservation_history[-1] - 100:.6f}%")

## 8. Visualisation de l'évolution (N+2 subplots verticaux)


In [ ]:
# Créer la figure avec N+2 subplots verticaux
fig, axes = plt.subplots(N_snapshots_total, 1, figsize=(10, 4 * N_snapshots_total))

# Échelle de couleur cohérente
vmax = max(b.max() for b in biomass_snapshots) / 10

for i, (biomass_snap, time_snap) in enumerate(zip(biomass_snapshots, time_snapshots)):
    # Appliquer le masque
    biomass_masked = biomass_snap * mask_global

    # Afficher la biomasse
    im = axes[i].imshow(
        biomass_masked,
        origin="lower",
        cmap="YlOrRd",
        vmin=0,
        vmax=vmax,
        interpolation="bilinear",
        extent=[0, global_nlon, 0, global_nlat],
    )

    # Contour de l'île
    axes[i].contour(
        mask_global,
        levels=[0.5],
        colors="black",
        linewidths=2,
        extent=[0, global_nlon, 0, global_nlat],
    )

    # Titre avec temps et conservation
    if i == 0:
        title = f"État Initial - t = {time_snap:.2f} h"
    elif i == N_snapshots_total - 1:
        title = f"État Final - t = {time_snap:.2f} h - Conservation: {conservation_history[i]:.4f}%"
    else:
        title = f"Intermédiaire {i} - t = {time_snap:.2f} h - Conservation: {conservation_history[i]:.4f}%"

    axes[i].set_title(title, fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Longitude (index)")
    axes[i].set_ylabel("Latitude (index)")

    # Colorbar
    plt.colorbar(im, ax=axes[i], label="Biomasse [kg/m²]", fraction=0.046)

plt.suptitle(
    "Évolution du blob de biomasse : Biologie + Transport avec île",
    fontsize=16,
    fontweight="bold",
    y=1.0,
)
plt.tight_layout()
plt.show()

## 9. Analyse de la conservation de la masse


In [ ]:
# Graphiques de conservation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Évolution de la masse
axes[0].plot(time_snapshots, np.array(mass_history) / 1e9, "b-o", linewidth=2, markersize=8)
axes[0].axhline(mass_init / 1e9, color="r", linestyle="--", linewidth=2, label="Masse initiale")
axes[0].set_xlabel("Temps [heures]", fontsize=12)
axes[0].set_ylabel("Masse totale [10⁹ kg]", fontsize=12)
axes[0].set_title("Évolution de la masse totale", fontsize=14, fontweight="bold")
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=10)

# Conservation
axes[1].plot(time_snapshots, conservation_history, "g-o", linewidth=2, markersize=8)
axes[1].axhline(100, color="r", linestyle="--", linewidth=2, label="Conservation parfaite")
axes[1].axhline(99, color="orange", linestyle=":", linewidth=2, label="Seuil 99%")
axes[1].set_xlabel("Temps [heures]", fontsize=12)
axes[1].set_ylabel("Conservation [%]", fontsize=12)
axes[1].set_title("Conservation de la masse", fontsize=14, fontweight="bold")
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n📊 Statistiques finales:")
print(f"  - Masse initiale : {mass_init:.2e} kg")
print(f"  - Masse finale : {mass_history[-1]:.2e} kg")
print(f"  - Conservation : {conservation_history[-1]:.6f}%")
print(f"  - Variation : {(conservation_history[-1] - 100):.6f}%")

## 10. Vérification du blocage de flux par l'île


In [ ]:
# Vérifier que l'île reste sans biomasse
final_biomass = biomass_snapshots[-1]
island_biomass = final_biomass[mask_global == 0]

print(f"Biomasse sur l'île (finale):")
print(f"  - Min: {island_biomass.min():.2e} kg/m²")
print(f"  - Max: {island_biomass.max():.2e} kg/m²")
print(f"  - Mean: {island_biomass.mean():.2e} kg/m²")

if np.all(island_biomass < 1e-6):
    print("\n✅ Le flux est correctement bloqué par l'île !")
else:
    print("\n⚠️ Attention : de la biomasse a pénétré l'île")

# Visualisation zoom sur l'île
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Biomasse finale globale
im1 = axes[0].imshow(final_biomass, origin="lower", cmap="YlOrRd", interpolation="bilinear")
axes[0].contour(mask_global, levels=[0.5], colors="black", linewidths=2)
axes[0].set_title("Biomasse Finale (globale)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Longitude (index)")
axes[0].set_ylabel("Latitude (index)")
plt.colorbar(im1, ax=axes[0], label="Biomasse [kg/m²]")

# Zoom sur l'île
zoom_size = 30
i_min, i_max = island_y - zoom_size, island_y + zoom_size
j_min, j_max = island_x - zoom_size, island_x + zoom_size

im2 = axes[1].imshow(
    final_biomass[i_min:i_max, j_min:j_max], origin="lower", cmap="YlOrRd", interpolation="bilinear"
)
axes[1].contour(mask_global[i_min:i_max, j_min:j_max], levels=[0.5], colors="black", linewidths=3)
axes[1].set_title("Zoom sur l'île", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Longitude (index relatif)")
axes[1].set_ylabel("Latitude (index relatif)")
plt.colorbar(im2, ax=axes[1], label="Biomasse [kg/m²]")

plt.tight_layout()
plt.show()

## 11. Résumé du système intégré

Ce notebook a démontré le système complet **Biologie + Transport distribué** :

### Architecture

-   **EventScheduler** : Coordonne les 5 phases (Biologie → Collecte → Transport → Redistribution → Agrégation)
-   **CellWorker2D** (4 workers) : Calculs biologiques distribués en parallèle
-   **TransportWorker** : Transport centralisé sur grille globale

### Physique

-   **Biologie** : Croissance exponentielle faible (r=0.0001)
-   **Advection** : Courant constant vers l'EST (u=0.2 m/s)
-   **Diffusion** : D=500 m²/s
-   **Masque** : Île centrale bloquant le flux
-   **Conditions limites** : CLOSED (flux nul aux bords)

### Résultats

-   ✅ Conservation de la masse > 99%
-   ✅ Flux correctement bloqué par l'île
-   ✅ Couplage biologie-transport fonctionnel
-   ✅ Workflow distribué opérationnel


In [ ]:
# Arrêter Ray (optionnel)
ray.shutdown()
# print("Ray arrêté")